In [ ]:
# Cell 1 — Clone repository and configure Python path
import os, sys

REPO_DIR = '/content/guardian-pixel'

if os.path.isdir(REPO_DIR):
    print('Repo already exists — pulling latest changes...')
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/GorohovskiMax/guardian-pixel.git {REPO_DIR}

%cd /content/guardian-pixel
sys.path.insert(0, '/content/guardian-pixel')
print('Working directory:', os.getcwd())

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive'
print(f'Drive mounted at: {DRIVE_ROOT}')
# ARTIFACT_ROOT is set dynamically in the next cell based on the zip structure

In [ ]:
# Cell 3 — Inspect zip contents, set ARTIFACT_ROOT, and unzip if needed
import os, zipfile

ZIP_PATH = '/content/drive/MyDrive/Kaggle_Datasets/artifact-dataset.zip'

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(f'Zip not found: {ZIP_PATH}')

# Step 1 — Print first 20 entries so we can verify the extracted folder name
print('First 20 entries in zip:')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    entries = zf.namelist()
    for entry in entries[:20]:
        print(f'  {entry}')
print(f'  ... ({len(entries):,} total files)')

# Step 2 — Auto-detect top-level folder name from zip entries
top_folders = {e.split('/')[0] for e in entries if '/' in e}
if len(top_folders) != 1:
    raise ValueError(
        f'Expected exactly 1 top-level folder in zip, found: {top_folders}\n'
        'Set ARTIFACT_ROOT manually and skip this cell.'
    )
extracted_folder = top_folders.pop()
ARTIFACT_ROOT = f'/content/drive/MyDrive/{extracted_folder}'
print(f'\nDetected folder  : {extracted_folder}')
print(f'ARTIFACT_ROOT    : {ARTIFACT_ROOT}')

# Step 3 — Unzip only if the dataset directory doesn't already exist
if os.path.isdir(ARTIFACT_ROOT):
    print('\nDataset already unzipped — skipping.')
else:
    print(f'\nUnzipping {ZIP_PATH} ...')
    total = len(entries)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        for i, member in enumerate(entries, 1):
            zf.extract(member, '/content/drive/MyDrive/')
            if i % 10000 == 0 or i == total:
                print(f'  Extracted {i:,} / {total:,} files...')
    print(f'Done. Dataset ready at {ARTIFACT_ROOT}')

In [ ]:
# Cell 4 — Install missing dependencies
!pip install -q wandb timm albumentations ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git
print('All dependencies installed.')

In [ ]:
# Cell 5 — Verify GPU availability and print device info
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU             : {props.name}')
    print(f'VRAM total      : {props.total_memory / 1e9:.1f} GB')
    print(f'CUDA version    : {torch.version.cuda}')
else:
    print('WARNING: No GPU detected. Training will be extremely slow.')

!nvidia-smi

In [ ]:
# Cell 6 — ForensicDetector sanity check
#
# Confirms:
#   1. FSR applied  — stem[0] stride should be (2, 2), not (4, 4)
#   2. Forward pass — output shape must be (1, 7) for the 7-class head

import torch
from layers.forensic import ForensicDetector

CONFIG = 'configs/layer_a.yaml'

model = ForensicDetector.from_config(CONFIG)

print('=== stem after FSR ===')
print(model.model.stem[0])
print()

dummy = torch.randn(1, 3, 200, 200)
model.eval()
with torch.no_grad():
    logits = model(dummy)

assert logits.shape == (1, 7), f'Expected (1, 7), got {tuple(logits.shape)}'
print(f'Forward pass output shape : {tuple(logits.shape)}  OK')

In [ ]:
# Cell 7 — Full training run
#
# Reads all hyperparameters from configs/layer_a.yaml.
# ARTIFACT_ROOT overrides data.root so the Drive path is used.
# W&B run name is read from logging.wandb_run_name in the config.

from layers.training import train

trained_model = train(
    config_path='configs/layer_a.yaml',
    data_root=ARTIFACT_ROOT,
    wandb_project='guardian-pixel',
)

In [ ]:
# Cell 8 — Load best checkpoint and evaluate on the test set

import torch
from layers.forensic import ForensicDetector
from layers.training import evaluate
from utils.dataloader import get_dataloaders

CONFIG     = 'configs/layer_a.yaml'
CHECKPOINT = 'models/layer_a/best.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Restore best weights
model = ForensicDetector.from_config(CONFIG).to(device)
checkpoint = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(
    f'Loaded checkpoint  epoch={checkpoint["epoch"]}  '
    f'val_bal_acc={checkpoint["best_balanced_accuracy"]:.4f}'
)

# Test set evaluation
loaders = get_dataloaders(CONFIG, root=ARTIFACT_ROOT)
test_metrics = evaluate(model, loaders['test'], device)

print()
print('=== Test Set Results ===')
print(f'  Loss              : {test_metrics["loss"]:.4f}')
print(f'  Accuracy          : {test_metrics["accuracy"]:.4f}')
print(f'  Balanced Accuracy : {test_metrics["balanced_accuracy"]:.4f}')